# Minimum Silver Table 4: QASMBench Circuits
**Course:** TU Delft DSAIT4000 (Data Management & Engineering) — Assignment 1  
**Target Table:** `silver/qasmbench/circuit.parquet`  
**Shared Provenance:** `results/part1/source_trace.parquet`


---

### Objectives & Contracts
1. **Schema Fidelity:** Build `silver/qasmbench/circuit.parquet` where **one row represents one parsed circuit variant**.
2. **Strict PyArrow Types:**
    - `source_record_id`: `string`  (stable link to the QASM member) 
    - `circuit_id`: `string` (stable circuit-variant identifier)
    - `benchmark_name` `string` (benchmark/circuit family)
    - `variant` `string` (source or transpiled variant)
    - `register_declarations` `string` (repeatable serialized register description)
    - `qubit_count`: `int32` (declared qubit count)
    - `measurement_count`: `int32` (executed measurement-operation count)
    - `two_qubit_gate_count`: `int32` (executed two-qubit-operation count)
3. **Shared Tracing Rule:** Append the QASMBenchmark circuit metadata records to `results/part1/source_trace.parquet` using the modular `save_source_traces`. 
4. **Dual-Lake Persistence:** Write Parquet to local disk and synchronize to MinIO bucket `quantum-lake`.


A QASMBench circuit row represents one parsed circuit variant. It captures the circuit’s benchmark identity, variant, register declarations, and overall structural counts: how many qubits are declared, how many measurement operations are executed after expanding register-based operations, and how many two-qubit gates are executed. This is summary metadata for the circuit, not a row-per-operation table.

**Background Definitions**

* **Circuit:** A quantum program or routine consisting of a sequence of operations on qubits that defines how a quantum computation is performed.
* **QASM Member:** A single file or source artifact in the QASMBench dataset that contains a QASM circuit description.
* **Variant:** A specific version or form of the same benchmark circuit.



In [1]:
import hashlib
import io
from pathlib import Path
import zipfile

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Platform helpers from starter package
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client
from quantum_lake_student.tracing import save_source_traces

# Detect if running in container (/workspace) or locally
BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
print(f"Base Directory: {BASE_DIR}")

# Load configuration
settings = Settings.from_environment()
print(f"Lake Backend: {settings.lake_backend}")
print(f"MinIO Endpoint: {settings.s3_endpoint} (Bucket: {settings.s3_bucket})")

Base Directory: /workspace
Lake Backend: minio
MinIO Endpoint: http://minio:9000 (Bucket: quantum-lake)


### Step 1: Bronze ingestion and deterministic parsing

The curated QASMBench archive contains three benchmark families, each with a source and transpiled OpenQASM variant. The parser below reads members in sorted order, removes comments and gate definitions from the executed stream, and expands register-wide operations when counting measurements and two-qubit gates.

In [2]:
# Locate the QASMBench Bronze zip object (try MinIO first, fallback to local path)
bronze_object_name = "bronze/source=qasmbench/qasmbench-qec.zip"
bronze_bytes = None

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        print(f"Fetching '{bronze_object_name}' from MinIO...")
        response = client.get_object(settings.s3_bucket, bronze_object_name)
        bronze_bytes = response.read()
        response.close()
        response.release_conn()
        print(f"Successfully retrieved from MinIO ({len(bronze_bytes):,} bytes)")
    except Exception as e:
        print(f"MinIO fetch warning: {e}. Falling back to local file.")

if bronze_bytes is None:
    candidates = [
        Path("/course-data/raw/source=qasmbench/qasmbench-qec.zip"),
        BASE_DIR.parent / "datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip",
        Path("datasets/student-bundle/core/raw/source=qasmbench/qasmbench-qec.zip"),
    ]
    for path in candidates:
        if path.exists():
            print(f"Reading from local path: {path}")
            bronze_bytes = path.read_bytes()
            break

assert bronze_bytes is not None, "Could not locate the QASMBench archive!"
bronze_sha256 = hashlib.sha256(bronze_bytes).hexdigest()
print(f"QASMBench Bronze Archive SHA-256: {bronze_sha256}")
print(f"QASMBench Bronze Archive Size:    {len(bronze_bytes):,} bytes")

Fetching 'bronze/source=qasmbench/qasmbench-qec.zip' from MinIO...
Successfully retrieved from MinIO (144,172 bytes)
QASMBench Bronze Archive SHA-256: 60307f88e34b1f752b94223d6d136da72c629b4b625ac6d1c30e5e2e4f85722a
QASMBench Bronze Archive Size:    144,172 bytes


### Step 2: Feature extraction and table column variable composiiton

In [3]:
import re

TWO_QUBIT_GATES = {"cx", "cnot", "cz", "ch", "cy", "swap", "cu1", "crz"}
REGISTER_DECLARATION = re.compile(r"^(qreg|creg)\s+(\w+)\s*\[\s*(\d+)\s*\]$")
GATE_HEADER = re.compile(r"gate\s+(\w+)\s*([^{}]*)\{(?P<body>.*?)\}", re.DOTALL)

def operand_width(operand, registers):
    match = re.fullmatch(r"(\w+)(?:\[(\d+)\])?", operand.strip())
    assert match, f"Unsupported operand: {operand}"
    name, index = match.groups()
    assert name in registers, f"Unknown register: {name}"
    return 1 if index is not None else registers[name]

def expanded_operation_count(operands, registers):
    widths = [operand_width(operand, registers) for operand in operands]
    assert widths and len(set(widths)) == 1, f"Register widths do not align: {operands}"
    return widths[0]

def body_two_qubit_count(body):
    count = 0
    for statement in body.split(";"):
        tokens = statement.strip().split(None, 1)
        if tokens and tokens[0].lower() in TWO_QUBIT_GATES:
            count += 1
    return count

def parse_qasm(member, text):
    cleaned = re.sub(r"//.*", "", text)
    gate_definitions = {}
    for match in GATE_HEADER.finditer(cleaned):
        gate_definitions[match.group(1)] = {
            "formal_count": len([item for item in match.group(2).split(",") if item.strip()]),
            "two_qubit_count": body_two_qubit_count(match.group("body")),
        }
    executable_text = GATE_HEADER.sub("", cleaned)

    registers = {}
    qreg_sizes = {}
    register_lines = []
    measurement_count = 0
    two_qubit_gate_count = 0
    for statement in executable_text.replace("{", " ").replace("}", " ").split(";"):
        statement = " ".join(statement.split())
        if not statement or statement.startswith(("OPENQASM", "include", "barrier", "opaque")):
            continue
        declaration = REGISTER_DECLARATION.fullmatch(statement)
        if declaration:
            kind, name, size = declaration.groups()
            registers[name] = int(size)
            if kind == "qreg":
                qreg_sizes[name] = int(size)
            register_lines.append(f"{kind} {name}[{size}]")
            continue
        if statement.startswith("measure "):
            left = statement[len("measure "):].split("->", 1)[0].strip()
            measurement_count += operand_width(left, registers)
            continue
        if statement.startswith("if("):
            statement = statement.split(")", 1)[1].strip()
        tokens = statement.split(None, 1)
        if len(tokens) != 2:
            continue
        gate_name, operand_text = tokens
        operands = [item.strip() for item in operand_text.split(",")]
        gate_key = gate_name.lower()
        if gate_key in TWO_QUBIT_GATES:
            two_qubit_gate_count += expanded_operation_count(operands, registers)
        elif gate_name in gate_definitions:
            widths = [operand_width(item, registers) for item in operands]
            assert len(widths) == gate_definitions[gate_name]["formal_count"]
            assert len(set(widths)) == 1, f"Custom gate widths do not align: {statement}"
            two_qubit_gate_count += gate_definitions[gate_name]["two_qubit_count"] * widths[0]

    benchmark_name = Path(member).parent.name
    filename = Path(member).stem
    variant = "transpiled" if filename.endswith("_transpiled") else "source"
    return {
        "source_record_id": f"qasmbench:{member}",
        "circuit_id": f"qasmbench:{member[:-5]}",
        "benchmark_name": benchmark_name,
        "variant": variant,
        "register_declarations": "; ".join(register_lines),
        "qubit_count": sum(qreg_sizes.values()),
        "measurement_count": measurement_count,
        "two_qubit_gate_count": two_qubit_gate_count,
    }

circuit_records = []
trace_records = []
with zipfile.ZipFile(io.BytesIO(bronze_bytes)) as archive:
    qasm_members = sorted(name for name in archive.namelist() if name.endswith(".qasm"))
    for member in qasm_members:
        record = parse_qasm(member, archive.read(member).decode("utf-8"))
        circuit_records.append(record)
        trace_records.append({
            "source_record_id": record["source_record_id"],
            "source_name": "qasmbench",
            "bronze_object": bronze_object_name,
            "archive_member": member,
            "record_locator": "qasm_program",
            "input_sha256": bronze_sha256,
        })
        print(f"  {record['benchmark_name']}/{record['variant']}: {record['qubit_count']} qubits, {record['measurement_count']} measurements, {record['two_qubit_gate_count']} two-qubit gates")

assert len(circuit_records) == 6
assert len({record["source_record_id"] for record in circuit_records}) == len(circuit_records)
print(f"\nParsed {len(circuit_records)} QASMBench circuit variants")

  error_correctiond3_n5/source: 5 qubits, 5 measurements, 49 two-qubit gates
  error_correctiond3_n5/transpiled: 5 qubits, 5 measurements, 49 two-qubit gates
  qec_en_n5/source: 5 qubits, 5 measurements, 10 two-qubit gates
  qec_en_n5/transpiled: 5 qubits, 5 measurements, 10 two-qubit gates
  qec_sm_n5/source: 5 qubits, 5 measurements, 4 two-qubit gates
  qec_sm_n5/transpiled: 5 qubits, 5 measurements, 4 two-qubit gates

Parsed 6 QASMBench circuit variants


### Step 3: Strict Arrow schema and Parquet export

The table stores one summary row per QASM member. The trace helper keeps this table's lineage alongside the traces produced by the other silver notebooks.

In [4]:
qasmbench_circuit_schema = pa.schema([
    ("source_record_id", pa.string()),
    ("circuit_id", pa.string()),
    ("benchmark_name", pa.string()),
    ("variant", pa.string()),
    ("register_declarations", pa.string()),
    ("qubit_count", pa.int32()),
    ("measurement_count", pa.int32()),
    ("two_qubit_gate_count", pa.int32()),
])

table_circuits = pa.Table.from_pandas(
    pd.DataFrame(circuit_records),
    schema=qasmbench_circuit_schema,
    preserve_index=False,
)
assert table_circuits.schema.equals(qasmbench_circuit_schema)
print("=== QASMBench Circuit Table ===")
print(f"Rows: {table_circuits.num_rows}, Columns: {table_circuits.num_columns}")
print(table_circuits.schema)

=== QASMBench Circuit Table ===
Rows: 6, Columns: 8
source_record_id: string
circuit_id: string
benchmark_name: string
variant: string
register_declarations: string
qubit_count: int32
measurement_count: int32
two_qubit_gate_count: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1098


### Step 4: Strict PyArrow Schema & Arrow Table Construction Details


In [5]:
# Define paths relative to the notebook's base workspace
silver_dir = BASE_DIR / "silver/qasmbench"
silver_dir.mkdir(parents=True, exist_ok=True)
silver_parquet_path = silver_dir / "circuit.parquet"

results_dir = BASE_DIR / "results/part1"
results_dir.mkdir(parents=True, exist_ok=True)
trace_parquet_path = results_dir / "source_trace.parquet"

pq.write_table(table_circuits, silver_parquet_path, compression="zstd")
print(f"Wrote Silver circuit table to: {silver_parquet_path} ({silver_parquet_path.stat().st_size:,} bytes)")

if settings.lake_backend == "minio":
    try:
        client = minio_client(settings)
        minio_key = "silver/qasmbench/circuit.parquet"
        client.fput_object(settings.s3_bucket, minio_key, str(silver_parquet_path))
        print(f"Uploaded Silver table to MinIO: {settings.s3_bucket}/{minio_key}")
    except Exception as e:
        print(f"Warning: MinIO upload failed: {e}")

trace_count = save_source_traces(
    new_records=trace_records,
    source_name="qasmbench",
    trace_file_path=trace_parquet_path,
    settings=settings,
)
print(f"Master source_trace table updated; total rows: {trace_count:,}")

saved = pq.read_table(silver_parquet_path)
assert saved.schema.equals(qasmbench_circuit_schema)
assert saved.num_rows == 6
print("Table 4 validation passed.")

Wrote Silver circuit table to: /workspace/silver/qasmbench/circuit.parquet (6,122 bytes)
Uploaded Silver table to MinIO: quantum-lake/silver/qasmbench/circuit.parquet
Master source_trace table updated; total rows: 6
Table 4 validation passed.


In [6]:
import pyarrow.parquet as pq

circuit_path = "/workspace/silver/qasmbench/circuit.parquet"
trace_path = "/workspace/results/part1/source_trace.parquet"

circuits = pq.read_table(circuit_path)
traces = pq.read_table(trace_path)

print("Circuit table")
print("Rows:", circuits.num_rows)
print("Columns:", circuits.column_names)
print(circuits.schema)
print(circuits.to_pandas().to_string(index=False))

print("\nQASMBench traces")
trace_df = traces.to_pandas()
qasmbench_traces = trace_df[trace_df["source_name"] == "qasmbench"]
print("Rows:", len(qasmbench_traces))
print(qasmbench_traces.to_string(index=False))

Circuit table
Rows: 6
Columns: ['source_record_id', 'circuit_id', 'benchmark_name', 'variant', 'register_declarations', 'qubit_count', 'measurement_count', 'two_qubit_gate_count']
source_record_id: string
circuit_id: string
benchmark_name: string
variant: string
register_declarations: string
qubit_count: int32
measurement_count: int32
two_qubit_gate_count: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1098
                                                           source_record_id                                                             circuit_id        benchmark_name    variant                        register_declarations  qubit_count  measurement_count  two_qubit_gate_count
           qasmbench:small/error_correctiond3_n5/error_correctiond3_n5.qasm            qasmbench:small/error_correctiond3_n5/error_correctiond3_n5 error_correctiond3_n5     source                         qreg q[5]; creg c[5]            5               

In [7]:
circuit_ids = set(circuits["source_record_id"].to_pylist())
trace_ids = set(qasmbench_traces["source_record_id"])

assert circuit_ids <= trace_ids
print("All circuit rows have source traces.")

All circuit rows have source traces.
